In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 23.9 MB/s  0:00:01


In [2]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/melting_point_project"   # поправьте под своё название папки
RAW = f"{BASE}/data/raw"
PROC = f"{BASE}/data/processed"

Mounted at /content/drive


In [3]:
import pandas as pd

In [4]:
X = pd.read_parquet(f"{PROC}/descriptors.parquet")      # или pd.read_csv(...)
meta = pd.read_csv(f"{PROC}/melting_points_clean.csv")
y = meta["mpC"].values
print(X.shape, len(y))

(19824, 215) 19824


При случайном разделении в тесте оказываются родственники обучающих молекул, и модель частично «узнаёт» их. Здесь в тесте остаются каркасы, которых модель не видела, то есть проверяется способность предсказывать для новых классов соединений. Это ближе к реальному использованию.

In [6]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit

In [11]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

**MurckoScaffold** cut side chain and return ramain smiles

In [7]:
scaf = meta['smiles_can'].apply(lambda s: MurckoScaffold.MurckoScaffoldSmiles(smiles=s))

Create groups with aromatic ring and others each molecules became group

In [8]:
group = [sc if sc else s for sc, s in zip(scaf, meta['smiles_can'])]

GroupShuffleSplit split data as molecules from one group complitely come into train or test

In [9]:
train, test = next(GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=0).split(X, y, group))

In [12]:
import numpy as np

med = np.median(y[train])                  # медиана по ОБУЧАЮЩЕЙ части
base = np.full(len(test), med)            # одно и то же число для каждой тестовой молекулы

print("медиана: MAE", mean_absolute_error(y[test], base),
      "RMSE", mean_squared_error(y[test], base) ** 0.5)

медиана: MAE 66.24015869980879 RMSE 82.4969913915555


In [14]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(n_estimators=50,n_jobs=-1, random_state=0)
)
model.fit(X.iloc[train], y[train])
pred = model.predict(X.iloc[test])
print("RF scaffold:      MAE", mean_absolute_error(y[test], pred),
      "RMSE", mean_squared_error(y[test], pred) ** 0.5)

RF scaffold:      MAE 38.25453379922616 RMSE 48.40141100935072
